In [1]:
spark

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,Current session?
0,application_1785476601588_0001,pyspark,idle,Link,Link,✔


SparkSession available as 'spark'.

In [2]:
# PySpark SQL Functions

from pyspark.sql.functions import (
    col,
    count,
    countDistinct,
    when,
    isnan,
    initcap,
    desc,
    asc,
    avg,
    sum,
    min,
    max,
    round,
    trim,
    lower,
    upper,
    split,
    explode,
    year,
    month,
    quarter,
    weekofyear,
    dayofmonth,
    datediff,
    current_date,
    lit,
    regexp_replace,
    create_map
)

from itertools import chain
from pyspark.sql.functions import *

# Window Functions

from pyspark.sql.window import Window


# PySpark Data Types

from pyspark.sql.types import (
    StringType,
    IntegerType,
    LongType,
    DateType,
    DoubleType
)

In [3]:
# =============================================================================
# Read Silver Song Charts
# =============================================================================

from pyspark.sql.functions import *

SILVER_SONG_PATH = "s3://group-1-dbda/silver/song_charts/"

silver_song_charts = spark.read.parquet(
    SILVER_SONG_PATH
)

silver_song_charts.printSchema()


root
 |-- date: date (nullable = true)
 |-- market: string (nullable = true)
 |-- rank: long (nullable = true)
 |-- uri: string (nullable = true)
 |-- artist_names: string (nullable = true)
 |-- track_name: string (nullable = true)
 |-- label: string (nullable = true)
 |-- peak_rank: long (nullable = true)
 |-- previous_rank: long (nullable = true)
 |-- days_on_chart: long (nullable = true)
 |-- streams: long (nullable = true)
 |-- consecutive_days: long (nullable = true)
 |-- entry_status: string (nullable = true)
 |-- peak_date: date (nullable = true)
 |-- entry_rank: long (nullable = true)
 |-- entry_date: date (nullable = true)
 |-- release_date: date (nullable = true)
 |-- artist_uris: string (nullable = true)
 |-- valid_release_date: date (nullable = true)
 |-- month: integer (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- week: integer (nullable = true)
 |-- song_age_days: integer (nullable = true)
 |-- song_age_category: string (nullable = true)
 |-- rank_movemen

In [4]:
# =============================================================================
# Business Dataset (Exclude Global Chart)
# =============================================================================

silver_song_charts_business = (

    silver_song_charts

    .filter(
        col("country_name") != "Global"
    )

)

In [6]:
print("Rows:", silver_song_charts_business.count())
print("Columns:", len(silver_song_charts_business.columns))

('Rows:', 42071431)
('Columns:', 32)

In [16]:
# =============================================================================
# Create Gold Song Charts
# =============================================================================

gold_song_charts = (

    silver_song_charts_business

    .select(

        "date",

        "year",

        "quarter",

        "month",

        "country_name",

        "uri",

        "track_name",

        "artist_uris",

        "artist_names",

        "standardized_label",

        "streams",

        "rank",

        "peak_rank",

        "days_on_chart",

        "song_age_category",

        "movement_category",

        "hit_category",

        "chart_strength_score",

        "stream_tier"

    )

)

In [17]:
gold_song_charts.printSchema()

root
 |-- date: date (nullable = true)
 |-- year: integer (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- country_name: string (nullable = true)
 |-- uri: string (nullable = true)
 |-- track_name: string (nullable = true)
 |-- artist_uris: string (nullable = true)
 |-- artist_names: string (nullable = true)
 |-- standardized_label: string (nullable = true)
 |-- streams: long (nullable = true)
 |-- rank: long (nullable = true)
 |-- peak_rank: long (nullable = true)
 |-- days_on_chart: long (nullable = true)
 |-- song_age_category: string (nullable = true)
 |-- movement_category: string (nullable = true)
 |-- hit_category: string (nullable = true)
 |-- chart_strength_score: double (nullable = true)
 |-- stream_tier: string (nullable = true)

In [18]:
print("Columns :", len(gold_song_charts.columns))

('Columns :', 19)

In [8]:
duplicate_rows = (

    gold_song_charts

    .groupBy(gold_song_charts.columns)

    .count()

    .filter(col("count") > 1)

    .count()

)

print("Duplicate Rows :", duplicate_rows)

('Duplicate Rows :', 0)

In [9]:
gold_song_charts.select(

    *[
        count(
            when(
                col(c).isNull(),
                c
            )
        ).alias(c)

        for c in gold_song_charts.columns
    ]

).show(vertical=True)

-RECORD 0-------------------
 date                 | 0   
 year                 | 0   
 quarter              | 0   
 month                | 0   
 country_name         | 0   
 uri                  | 0   
 track_name           | 0   
 artist_uris          | 0   
 artist_names         | 0   
 standardized_label   | 0   
 streams              | 0   
 rank                 | 0   
 peak_rank            | 0   
 days_on_chart        | 0   
 song_age_category    | 0   
 movement_category    | 0   
 hit_category         | 0   
 chart_strength_score | 0   
 stream_tier          | 0

In [10]:
gold_song_charts.select("year").distinct().orderBy("year").show()

+----+
|year|
+----+
|2017|
|2018|
|2019|
|2020|
|2021|
|2022|
|2023|
|2024|
|2025|
|2026|
+----+

In [19]:
print("Rows:", gold_song_charts.count())

('Rows:', 42071431)

In [20]:
# =============================================================================
# Optimize Gold Song Charts
# =============================================================================

gold_song_charts_optimized = (

    gold_song_charts

    .repartition(
        20,
        "year"
    )

)

# =============================================================================
# Write Gold Song Charts
# =============================================================================

(
    gold_song_charts_optimized

    .write

    .mode("overwrite")

    .option("compression", "snappy")

    .partitionBy(
        "year"
    )

    .parquet(
        "s3://group-1-dbda/gold/song_charts/"
    )

)

In [21]:
gold_song_charts_check = spark.read.parquet(
    "s3://group-1-dbda/gold/song_charts/"
)

print("Columns :", len(gold_song_charts_check.columns))

gold_song_charts_check.printSchema()

gold_song_charts_check.show(5, truncate=False)

('Columns :', 19)
root
 |-- date: date (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- country_name: string (nullable = true)
 |-- uri: string (nullable = true)
 |-- track_name: string (nullable = true)
 |-- artist_uris: string (nullable = true)
 |-- artist_names: string (nullable = true)
 |-- standardized_label: string (nullable = true)
 |-- streams: long (nullable = true)
 |-- rank: long (nullable = true)
 |-- peak_rank: long (nullable = true)
 |-- days_on_chart: long (nullable = true)
 |-- song_age_category: string (nullable = true)
 |-- movement_category: string (nullable = true)
 |-- hit_category: string (nullable = true)
 |-- chart_strength_score: double (nullable = true)
 |-- stream_tier: string (nullable = true)
 |-- year: integer (nullable = true)

+----------+-------+-----+------------+------------------------------------+-----------------------------------------+-------------------------------------------------------------

Visual 2

In [47]:
# =============================================================================
# Base Country Performance
# =============================================================================

country_performance = (

    silver_song_charts_business

    .groupBy(

        "year",
        "month",
        "country_name"

    )

    .agg(

        round(

            sum("streams"),

            0

        ).alias(

            "total_streams"

        ),

        countDistinct(

            "uri"

        ).alias(

            "active_songs"

        ),

        countDistinct(

            when(

                col("hit_category").isin(

                    "Global Hit",
                    "Major Hit"

                ),

                col("uri")

            )

        ).alias(

            "hit_songs"

        ),

        round(

            avg("chart_strength_score"),

            2

        ).alias(

            "avg_chart_strength"

        )

    )

)

In [48]:
# =============================================================================
# Active Artists
# =============================================================================

country_artists = (

    silver_song_charts_business

    .select(

        "year",
        "month",
        "country_name",

        explode(

            split(

                col("artist_uris"),

                "\\|"

            )

        ).alias(

            "artist_uri"

        )

    )

    .withColumn(

        "artist_uri",

        trim(

            col("artist_uri")

        )

    )

    .filter(

        col("artist_uri") != ""

    )

    .groupBy(

        "year",
        "month",
        "country_name"

    )

    .agg(

        countDistinct(

            "artist_uri"

        ).alias(

            "active_artists"

        )

    )

)

In [49]:
# =============================================================================
# Merge Active Artists
# =============================================================================

country_performance = (

    country_performance

    .join(

        country_artists,

        [

            "year",
            "month",
            "country_name"

        ],

        "left"

    )

)

In [50]:
# =============================================================================
# Monthly Total Streams
# =============================================================================

monthly_streams = (

    country_performance

    .groupBy(

        "year",
        "month"

    )

    .agg(

        sum(

            "total_streams"

        ).alias(

            "monthly_total_streams"

        )

    )

)

In [51]:
# =============================================================================
# Merge Monthly Total Streams
# =============================================================================

country_performance = (

    country_performance

    .join(

        monthly_streams,

        [

            "year",
            "month"

        ],

        "left"

    )

)

In [52]:
country_performance.printSchema()

root
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- country_name: string (nullable = true)
 |-- total_streams: long (nullable = true)
 |-- active_songs: long (nullable = false)
 |-- hit_songs: long (nullable = false)
 |-- avg_chart_strength: double (nullable = true)
 |-- active_artists: long (nullable = true)
 |-- monthly_total_streams: long (nullable = true)

In [28]:
print(
    "Rows:",
    country_performance.count()
)

('Rows:', 7383)

In [53]:
country_song_streams = (

    silver_song_charts_business

    .filter(
        col("track_name") != "Unknown Track"
    )

    .groupBy(

        "year",
        "month",
        "country_name",
        "track_name"

    )

    .agg(

        sum("streams").alias(
            "song_streams"
        )

    )

)

song_window = Window.partitionBy(

    "year",
    "month",
    "country_name"

).orderBy(

    desc("song_streams"),
    asc("track_name")

)

top_song = (

    country_song_streams

    .withColumn(

        "rank",

        row_number().over(song_window)

    )

    .filter(
        col("rank") == 1
    )

    .select(

        "year",
        "month",
        "country_name",

        col("track_name").alias(
            "top_song_name"
        ),

        "song_streams"

    )

)

In [54]:
country_performance = (

    country_performance

    .join(

        top_song.drop("song_streams"),

        [
            "year",
            "month",
            "country_name"
        ],

        "left"

    )

)

In [55]:
# =============================================================================
# Explode Artists
# =============================================================================

country_artist_streams = (

    silver_song_charts_business

    .select(

        "year",
        "month",
        "country_name",
        "streams",

        explode(

            arrays_zip(

                split(col("artist_names"), "\\|"),

                split(col("artist_uris"), "\\|")

            )

        ).alias("artist")

    )

    .select(

        "year",
        "month",
        "country_name",
        "streams",

        trim(
            col("artist.0")
        ).alias("artist_name"),

        trim(
            col("artist.1")
        ).alias("artist_uri")

    )

    .filter(
        col("artist_name") != "Unknown Artist"
    )

)

In [56]:
country_artist_streams = (

    country_artist_streams

    .groupBy(

        "year",
        "month",
        "country_name",
        "artist_name",
        "artist_uri"

    )

    .agg(

        sum("streams").alias(
            "artist_streams"
        )

    )

)

In [57]:
artist_window = Window.partitionBy(

    "year",
    "month",
    "country_name"

).orderBy(

    desc("artist_streams"),
    asc("artist_name")

)

top_artist = (

    country_artist_streams

    .withColumn(

        "rank",

        row_number().over(
            artist_window
        )

    )

    .filter(
        col("rank") == 1
    )

    .select(

        "year",
        "month",
        "country_name",

        col("artist_name").alias(
            "top_artist_name"
        ),

        "artist_streams"

    )

)

In [58]:
# =============================================================================
# Merge Top Artist
# =============================================================================

country_performance = (

    country_performance

    .join(

        top_artist.drop("artist_streams"),

        [
            "year",
            "month",
            "country_name"
        ],

        "left"

    )

)

In [60]:
growth_window = (

    Window

    .partitionBy("country_name")

    .orderBy(
        "year",
        "month"
    )

)

country_performance = (

    country_performance

    .withColumn(

        "previous_month_streams",

        lag(
            "total_streams"
        ).over(
            growth_window
        )

    )

    .withColumn(

        "growth_percentage",

        round(

            when(

                col("previous_month_streams").isNull(),

                None

            )

            .otherwise(

                (

                    (
                        col("total_streams")
                        -
                        col("previous_month_streams")
                    )

                    /

                    col("previous_month_streams")

                ) * 100

            ),

            2

        )

    )

    .drop(
        "previous_month_streams"
    )

)

In [61]:
country_performance.printSchema()

root
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- country_name: string (nullable = true)
 |-- total_streams: long (nullable = true)
 |-- active_songs: long (nullable = false)
 |-- hit_songs: long (nullable = false)
 |-- avg_chart_strength: double (nullable = true)
 |-- active_artists: long (nullable = true)
 |-- monthly_total_streams: long (nullable = true)
 |-- top_song_name: string (nullable = true)
 |-- top_artist_name: string (nullable = true)
 |-- growth_percentage: double (nullable = true)

In [62]:
(
    country_performance

    .repartition(
        20,
        "year"
    )

    .write

    .mode("overwrite")

    .option(
        "compression",
        "snappy"
    )

    .partitionBy(
        "year"
    )

    .parquet(
        "s3://group-1-dbda/gold/country_performance/"
    )

)

In [63]:
COUNTRY_PERFORMANCE_PATH = "s3://group-1-dbda/gold/country_performance/"

country_performance = spark.read.parquet(
    COUNTRY_PERFORMANCE_PATH
)

In [64]:
print("Rows :", country_performance.count())

('Rows :', 7383)

In [65]:
country_performance.printSchema()

root
 |-- month: integer (nullable = true)
 |-- country_name: string (nullable = true)
 |-- total_streams: long (nullable = true)
 |-- active_songs: long (nullable = true)
 |-- hit_songs: long (nullable = true)
 |-- avg_chart_strength: double (nullable = true)
 |-- active_artists: long (nullable = true)
 |-- monthly_total_streams: long (nullable = true)
 |-- top_song_name: string (nullable = true)
 |-- top_artist_name: string (nullable = true)
 |-- growth_percentage: double (nullable = true)
 |-- year: integer (nullable = true)

Visual - 3 Artist performance

In [7]:
# =============================================================================
# Explode Artists
# =============================================================================

artist_data = (

    silver_song_charts_business

    .select(

        "year",
        "month",
        "country_name",
        "uri",
        "streams",
        "chart_strength_score",
        "hit_category",
        "artist_uris",
        "artist_names"

    )

    .withColumn(

        "artist",

        explode(

            arrays_zip(

                split(col("artist_uris"), "\\|"),

                split(col("artist_names"), "\\|")

            )

        )

    )

    .withColumn(
        "artist_uri",
        trim(col("artist.0"))
    )

    .withColumn(
        "artist_name",
        trim(col("artist.1"))
    )

    .drop(
        "artist",
        "artist_uris",
        "artist_names"
    )

    .filter(
        (col("artist_name").isNotNull()) &
        (trim(col("artist_name")) != "") &
        (col("artist_name") != "Unknown Artist")
    )

)

In [9]:
# =============================================================================
# Artist Performance
# =============================================================================

artist_performance = (

    artist_data

    .groupBy(

        "year",
        "month",
        "country_name",
        "artist_uri"

    )

    .agg(

        first(
            "artist_name",
            ignorenulls=True
        ).alias(
            "artist_name"
        ),

        round(

            sum("streams"),

            0

        ).alias(
            "total_streams"
        ),

        countDistinct(
            "uri"
        ).alias(
            "active_songs"
        ),

        countDistinct(

            when(

                col("hit_category").isin(
                    "Global Hit",
                    "Major Hit"
                ),

                col("uri")

            )

        ).alias(
            "hit_songs"
        ),

        round(

            avg(
                "chart_strength_score"
            ),

            2

        ).alias(
            "avg_chart_strength"
        )

    )

)

In [10]:
# =============================================================================
# Write Artist Performance
# =============================================================================

(
    artist_performance

    .repartition(
        20,
        "year"
    )

    .write

    .mode("overwrite")

    .option(
        "compression",
        "snappy"
    )

    .partitionBy(
        "year"
    )

    .parquet(
        "s3://group-1-dbda/gold/artist_performance/"
    )

)

In [11]:
# =============================================================================
# Read Artist Performance
# =============================================================================

ARTIST_PERFORMANCE_PATH = "s3://group-1-dbda/gold/artist_performance/"

artist_performance = spark.read.parquet(
    ARTIST_PERFORMANCE_PATH
)

In [12]:
artist_performance.printSchema()

print("Columns :", len(artist_performance.columns))
print("Rows :", artist_performance.count())

root
 |-- month: integer (nullable = true)
 |-- country_name: string (nullable = true)
 |-- artist_uri: string (nullable = true)
 |-- artist_name: string (nullable = true)
 |-- total_streams: long (nullable = true)
 |-- active_songs: long (nullable = true)
 |-- hit_songs: long (nullable = true)
 |-- avg_chart_strength: double (nullable = true)
 |-- year: integer (nullable = true)

('Columns :', 9)
('Rows :', 1892077)

Trend Analysis

In [21]:
# =============================================================================
# Monthly Trends
# =============================================================================

monthly_trends = (

    silver_song_charts_business

    .filter(

        ~(
            (col("year") == 2026) &
            (col("month") == 5)
        )

    )

    .groupBy(

        "year",
        "month",
        "country_name"

    )

    .agg(

        round(

            sum("streams"),

            0

        ).alias(
            "total_streams"
        ),

        countDistinct(
            "uri"
        ).alias(
            "active_songs"
        ),

        countDistinct(
            "standardized_label"
        ).alias(
            "active_labels"
        ),

        countDistinct(

            when(

                col("hit_category").isin(
                    "Global Hit",
                    "Major Hit"
                ),

                col("uri")

            )

        ).alias(
            "hit_songs"
        ),

        round(

            avg(
                "chart_strength_score"
            ),

            2

        ).alias(
            "avg_chart_strength"
        )

    )

)

In [22]:
# =============================================================================
# Monthly Active Artists
# =============================================================================

monthly_artists = (

    silver_song_charts_business

    .filter(

        ~(
            (col("year") == 2026) &
            (col("month") == 5)
        )

    )

    .select(

        "year",
        "month",
        "country_name",

        explode(

            split(
                col("artist_uris"),
                "\\|"
            )

        ).alias(
            "artist_uri"
        )

    )

    .withColumn(

        "artist_uri",

        trim(
            col("artist_uri")
        )

    )

    .filter(

        col("artist_uri") != ""

    )

    .groupBy(

        "year",
        "month",
        "country_name"

    )

    .agg(

        countDistinct(
            "artist_uri"
        ).alias(
            "active_artists"
        )

    )

)

In [23]:
# =============================================================================
# Join Active Artists
# =============================================================================

monthly_trends = (

    monthly_trends

    .join(

        monthly_artists,

        [

            "year",
            "month",
            "country_name"

        ],

        "left"

    )

)

In [24]:
# =============================================================================
# Growth Percentage
# =============================================================================

growth_window = (

    Window

    .partitionBy(

        "country_name"

    )

    .orderBy(

        "year",
        "month"

    )

)

In [25]:
monthly_trends = (

    monthly_trends

    .withColumn(

        "previous_month_streams",

        lag(

            "total_streams"

        ).over(

            growth_window

        )

    )

    .withColumn(

        "growth_percentage",

        when(

            (col("previous_month_streams").isNull()) |
            (col("previous_month_streams") == 0),

            None

        )

        .otherwise(

            round(

                (

                    (

                        col("total_streams")
                        -
                        col("previous_month_streams")

                    )

                    /

                    col("previous_month_streams")

                )

                * 100,

                2

            )

        )

    )

    .drop(

        "previous_month_streams"

    )

)

In [26]:
monthly_trends.printSchema()

root
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- country_name: string (nullable = true)
 |-- total_streams: long (nullable = true)
 |-- active_songs: long (nullable = false)
 |-- active_labels: long (nullable = false)
 |-- hit_songs: long (nullable = false)
 |-- avg_chart_strength: double (nullable = true)
 |-- active_artists: long (nullable = true)
 |-- growth_percentage: double (nullable = true)

In [27]:
monthly_trends.printSchema()

print("Columns :", len(monthly_trends.columns))
print("Rows :", monthly_trends.count())

root
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- country_name: string (nullable = true)
 |-- total_streams: long (nullable = true)
 |-- active_songs: long (nullable = false)
 |-- active_labels: long (nullable = false)
 |-- hit_songs: long (nullable = false)
 |-- avg_chart_strength: double (nullable = true)
 |-- active_artists: long (nullable = true)
 |-- growth_percentage: double (nullable = true)

('Columns :', 10)
('Rows :', 7313)

In [28]:
# =============================================================================
# Write Monthly Trends
# =============================================================================

(
    monthly_trends

    .coalesce(1)

    .write

    .mode("overwrite")

    .option(
        "compression",
        "snappy"
    )

    .parquet(
        "s3://group-1-dbda/gold/monthly_trends/"
    )

)

In [29]:
# =============================================================================
# Read Monthly Trends
# =============================================================================

MONTHLY_TRENDS_PATH = "s3://group-1-dbda/gold/monthly_trends/"

monthly_trends = spark.read.parquet(
    MONTHLY_TRENDS_PATH
)

In [30]:
print("Columns :", len(monthly_trends.columns))
print("Rows :", monthly_trends.count())

('Columns :', 10)
('Rows :', 7313)

Final Visual - Artist Country matrix

In [10]:
# =============================================================================
# Artist Country Dominance
# =============================================================================

artist_data = (

    silver_song_charts_business

    .select(

        "year",
        "month",
        "country_name",
        "uri",
        "streams",
        "chart_strength_score",
        "hit_category",
        "artist_uris",
        "artist_names"

    )

    .withColumn(

        "artist",

        explode(

            arrays_zip(

                split(
                    col("artist_uris"),
                    "\\|"
                ),

                split(
                    col("artist_names"),
                    "\\|"
                )

            )

        )

    )

    .withColumn(

        "artist_uri",

        trim(
            col("artist.0")
        )

    )

    .withColumn(

        "artist_name",

        trim(
            col("artist.1")
        )

    )

    .drop(

        "artist",
        "artist_uris",
        "artist_names"

    )

    .filter(

        (col("artist_uri").isNotNull()) &
        (trim(col("artist_uri")) != "") &

        (col("artist_name").isNotNull()) &
        (trim(col("artist_name")) != "") &
        (col("artist_name") != "Unknown Artist")

    )

    .dropDuplicates(

        [

            "year",
            "month",
            "country_name",
            "uri",
            "artist_uri"

        ]

    )

)

# =============================================================================
# Artist Country Performance
# =============================================================================

artist_country_performance = (

    artist_data

    .groupBy(

        "year",
        "month",
        "country_name",
        "artist_uri"

    )

    .agg(

        min(
            "artist_name"
        ).alias(
            "artist_name"
        ),

        round(

            sum(
                "streams"
            ),

            0

        ).alias(
            "artist_streams"
        ),

        countDistinct(
            "uri"
        ).alias(
            "active_songs"
        ),

        countDistinct(

            when(

                col("hit_category").isin(

                    "Global Hit",
                    "Major Hit"

                ),

                col("uri")

            )

        ).alias(
            "hit_songs"
        ),

        round(

            avg(
                "chart_strength_score"
            ),

            2

        ).alias(
            "avg_chart_strength"
        )

    )

)

In [11]:
# =============================================================================
# Country Total Streams
# =============================================================================

country_totals = (

    silver_song_charts_business

    .groupBy(

        "year",
        "month",
        "country_name"

    )

    .agg(

        round(

            sum("streams"),

            0

        ).alias(
            "country_total_streams"
        )

    )

)

In [12]:
# =============================================================================
# Market Share
# =============================================================================

artist_country_performance = (

    artist_country_performance

    .join(

        country_totals,

        [

            "year",
            "month",
            "country_name"

        ],

        "left"

    )

    .withColumn(

        "artist_market_share",

        round(

            when(

                col("country_total_streams") > 0,

                (
                    col("artist_streams")
                    /
                    col("country_total_streams")
                ) * 100

            )

            .otherwise(0),

            2

        )

    )

)

In [13]:
# =============================================================================
# Catalog Hit Rate
# =============================================================================

artist_country_performance = (

    artist_country_performance

    .withColumn(

        "catalog_hit_rate",

        round(

            when(

                col("active_songs") > 0,

                (
                    col("hit_songs")
                    /
                    col("active_songs")
                ) * 100

            )

            .otherwise(0),

            2

        )

    )

)

In [14]:
artist_country_performance = (

    artist_country_performance

    .select(

        "year",
        "month",
        "country_name",

        "artist_uri",
        "artist_name",

        "artist_streams",
        "country_total_streams",
        "artist_market_share",

        "active_songs",
        "hit_songs",
        "catalog_hit_rate",
        "avg_chart_strength"

    )

    .orderBy(

        "year",
        "month",
        "country_name",
        desc("artist_streams")

    )

)

In [15]:
artist_country_performance.printSchema()

root
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- country_name: string (nullable = true)
 |-- artist_uri: string (nullable = true)
 |-- artist_name: string (nullable = true)
 |-- artist_streams: long (nullable = true)
 |-- country_total_streams: long (nullable = true)
 |-- artist_market_share: double (nullable = true)
 |-- active_songs: long (nullable = false)
 |-- hit_songs: long (nullable = false)
 |-- catalog_hit_rate: double (nullable = true)
 |-- avg_chart_strength: double (nullable = true)

In [16]:
print('Rows : ', artist_country_performance.count())
print('Columns : ', len(artist_country_performance.columns))

('Rows : ', 1892069)
('Columns : ', 12)

In [17]:
(
    artist_country_performance

    .repartition(
        20,
        "year"
    )

    .write

    .mode("overwrite")

    .option(
        "compression",
        "snappy"
    )

    .partitionBy(
        "year"
    )

    .parquet(
        "s3://group-1-dbda/gold/artist_country_dominance/"
    )

)

In [18]:
artist_country_dominance = spark.read.parquet(
    "s3://group-1-dbda/gold/artist_country_dominance/"
)

In [19]:
print("Rows :", artist_country_dominance.count())
print("Columns :", len(artist_country_dominance.columns))

artist_country_dominance.printSchema()

('Rows :', 1892069)
('Columns :', 12)
root
 |-- month: integer (nullable = true)
 |-- country_name: string (nullable = true)
 |-- artist_uri: string (nullable = true)
 |-- artist_name: string (nullable = true)
 |-- artist_streams: long (nullable = true)
 |-- country_total_streams: long (nullable = true)
 |-- artist_market_share: double (nullable = true)
 |-- active_songs: long (nullable = true)
 |-- hit_songs: long (nullable = true)
 |-- catalog_hit_rate: double (nullable = true)
 |-- avg_chart_strength: double (nullable = true)
 |-- year: integer (nullable = true)